## Import libraries

In [ ]:
from pathlib import Path
import bioread
import numpy as np
import pandas as pd
from scipy import signal
from helpers import (compare, fir_bandpass, comb_band, integration_moving_rms,
                    read_acq_files, show_channels, choose_indexes_per_file, get_reference_channels)

## Load files

#### Path for the files

In [2]:
folder_root_path = r"data/"     # r"I:/Research/Shares/dover_research/Div_fEMG/"

In [3]:
study_id = "TEST"

In [4]:
files_suffix = {
    "APP": Path(f"{folder_root_path}{study_id}_fEMG_div_2APP_raw.acq"),
    "PsychoPy": Path(f"{folder_root_path}{study_id}_fEMG_div_3PsychoPy_raw.acq"),
    "SlowRate": Path(f"{folder_root_path}{study_id}_fEMG_div_4SlowRate_raw.acq"),
}

In [ ]:
if pd.DataFrame([
    {"task": task, "path": str(path.resolve()), "exists": path.is_file()}
    for task, path in files_suffix.items()
])['exists'].all():
    print('All file path exits')
else:
    print('Not all file path exist, please check rootfolder and file paths')

All file path exits


### Read files

In [7]:
data = read_acq_files(files_suffix)

In [9]:
data

{'APP': AcqKnowledge file (rev 133): 8 channels, 2000.0 samples/sec,
 'PsychoPy': AcqKnowledge file (rev 133): 8 channels, 2000.0 samples/sec,
 'SlowRate': AcqKnowledge file (rev 133): 5 channels, 2000.0 samples/sec}

In [10]:
data_df = show_channels(data)
data_df

,task,index,order_num,name,units,rate_hz,samples
0,APP,0,1,ZM - FIR,mV,2000.0,1262646
1,APP,1,2,ZM - Comb,mV,2000.0,1262646
2,APP,2,3,ZM - EMG100C,mV,2000.0,1262646
3,APP,3,4,CS - EMG100C,mV,2000.0,1262646
4,APP,4,5,ZM - Int,mV,2000.0,1262646
5,APP,5,6,CS - FIR,mV,2000.0,1262646
6,APP,6,7,CS - Comb,mV,2000.0,1262646
7,APP,7,8,CS - Integrate,mV,2000.0,1262646
8,PsychoPy,0,1,ZM - FIR,mV,2000.0,1137980
9,PsychoPy,1,2,ZM - Comb,mV,2000.0,1137980


In [69]:
app_idx = choose_indexes_per_file(data_df, "SlowRate") # "APP", "PsychoPy" or "SlowRate"
app_idx

{'ZM - FIR': 0,
 'ZM - Comb': 1,
 'ZM - EMG100C': 2,
 'CS - EMG100C': 3,
 'ZM - Integrate': 4}

In [14]:
app_idx["ZM - EMG100C"]

2

In [70]:
reference_channels = get_reference_channels(data, "SlowRate", app_idx, "ZM")

In [71]:
reference_channels

(Channel ZM - EMG100C: 805412 samples, 2000.0 samples/sec, loaded: True,
 Channel ZM - FIR: 805412 samples, 2000.0 samples/sec, loaded: True,
 Channel ZM - Comb: 805412 samples, 2000.0 samples/sec, loaded: True,
 Channel ZM - Integrate: 805412 samples, 2000.0 samples/sec, loaded: True)

In [72]:
sampling_rate =reference_channels[0].samples_per_second
sampling_rate

2000.0

In [73]:
fir = fir_bandpass(np.asarray(reference_channels[0].data, dtype=float), sampling_rate)
comb = comb_band(fir, sampling_rate, base_hz=60.0, q=5)
rms = integration_moving_rms(comb, window_samples=1000)

In [74]:
compare(fir, np.asarray(reference_channels[1].data, dtype=float), sampling_rate)

max_absolute_error       1.332268e-15
rmse                     1.194635e-16
relative_rmse_percent    5.664890e-14
mean_error              -5.188245e-20
worst_sample             6.206740e+05
worst_time_seconds       3.103370e+02
dtype: float64

In [75]:
compare(comb, np.asarray(reference_channels[2].data, dtype=float), sampling_rate)

max_absolute_error       6.577638e-15
rmse                     1.267564e-15
relative_rmse_percent    5.324635e-11
mean_error               4.213131e-18
worst_sample             6.164360e+05
worst_time_seconds       3.082180e+02
dtype: float64

In [76]:
compare(rms, np.asarray(reference_channels[3].data, dtype=float), sampling_rate)

max_absolute_error       1.026956e-15
rmse                     1.165339e-16
relative_rmse_percent    3.773983e-12
mean_error              -1.841143e-17
worst_sample             3.200000e+01
worst_time_seconds       1.600000e-02
dtype: float64